In [19]:
from dotenv import load_dotenv
import anthropic
import base64
import os

load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

model = "claude-opus-4-5"

print("Client ready. Model:", model)
print("API Key loaded:", os.getenv("ANTHROPIC_API_KEY")[:10], "...")  # verify key loaded

Client ready. Model: claude-opus-4-5
API Key loaded: sk-ant-api ...


In [20]:
def add_user_message(messages, message):
    messages.append({"role": "user", "content": message})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None):
    kwargs = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response

def text_from_message(response):
    return response.content[0].text

print("Helper functions defined.")

Helper functions defined.


In [21]:
prompt = """
Analyze the attached PDF document with these specific steps:

1. Document overview: Identify the document type and main subject:
   - What kind of document is this (report, contract, invoice, research paper, manual, etc.)?
   - What is the primary topic or purpose?
   - How many pages or sections does it appear to have?

2. Key content extraction: Identify the most important information:
   - List the main topics or sections covered
   - Extract any key figures, dates, names, or values mentioned
   - Note any conclusions, recommendations, or action items

3. Structure analysis: Describe the document's organization:
   - How is the content structured (headings, tables, bullet points)?
   - Are there any charts, images, or diagrams?
   - Note any appendices or supplementary sections

4. Summary: Provide a concise summary:
   - Write a 2-3 sentence summary of the entire document
   - Highlight the single most important takeaway

For each item (1-4), write one to two sentences summarizing findings.
Final response: a brief executive summary of the document in plain language.
"""

print("Prompt defined.")

Prompt defined.


In [22]:
def analyze_pdf_file(pdf_path, prompt):
    with open(pdf_path, "rb") as f:
        pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")

    response = client.messages.create(
        model=model,
        max_tokens=4000,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",        # Anthropic native PDF support
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_data
                    }
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }]
    )
    return text_from_message(response)

print("analyze_pdf_file() defined.")

analyze_pdf_file() defined.


In [ ]:
pdf_path = "earth.pdf"   # <- change filename as needed

print(f"\n{'='*50}")
print(f"Analyzing: {pdf_path}")
print('='*50)

result = analyze_pdf_file(pdf_path, prompt)
print(result)

In [ ]:
pdfs_folder = "pdfs"   # <- "." for current folder, or subfolder name

if not os.path.exists(pdfs_folder):
    print(f"Folder '{pdfs_folder}' not found. Creating it...")
    os.makedirs(pdfs_folder)
    print("Add your PDF files to the folder and re-run this cell.")
else:
    pdf_files = [f for f in os.listdir(pdfs_folder) if f.lower().endswith(".pdf")]
    print(f"Found {len(pdf_files)} PDFs: {pdf_files}")

    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdfs_folder, pdf_file)
        print(f"\n{'='*50}")
        print(f"Analyzing: {pdf_file}")
        print('='*50)
        result = analyze_pdf_file(pdf_path, prompt)
        print(result)

In [ ]:
def analyze_pdf_url(pdf_url, prompt):
    response = client.messages.create(
        model=model,
        max_tokens=4000,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "url",         # direct URL, no base64 needed
                        "url": pdf_url
                    }
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }]
    )
    return text_from_message(response)

# Test with a sample public PDF
url = "https://www.w3.org/WAI/WCAG21/wcag21.pdf"
result = analyze_pdf_url(url, prompt)
print(result)